In [ ]:
import geopandas as gpd
from tqdm import tqdm

In [ ]:
%%time

# Abre um arquivo e transforma em um geodataframe
gdf = gpd.read_parquet(r'E:\arquivos_modulo_5\mapbiomas_alerta.parquet')
# Filtra o Geodataframe pelos anos maiores que 2020
gdf = gdf[gdf['ANODETEC'] >= 2020]
# Declarando uma pasta como uma variável
input_folder = r'E:\arquivos_modulo_5\car_x_mpb'

In [23]:
import os

# Listando os arquivos de uma pasta
files = os.listdir(input_folder)

# Inicio um loop
for item in tqdm(files):
    # Transformando em geodataframe cada um dos arquivos de uma lsita
    current_geometry = gpd.read_file(os.path.join(input_folder,item))
    # Deletando uma coluna do geodataframe
    current_geometry = current_geometry.drop(columns='index_right')
    # Junção espacial dos dois geodataframes
    joined = gpd.sjoin(gdf,current_geometry,how="inner",predicate="intersects")
    # Formando o nome do arquivo de saída
    filename = f"{os.path.splitext(item)[0]}_desmat.geojson"
    # Checo se o goedataframe está vazio, se não estiver, salvo o arquivo
    if joined.empty:
        current_geometry.to_file(os.path.join(r'E:\arquivos_modulo_5\car_mpb_desmat',filename),driver='GeoJSON')


  0%|          | 0/7697 [00:00<?, ?it/s]C:\Users\Vitor\AppData\Local\Temp\ipykernel_11112\1367202915.py:11: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: OGC:CRS84
Right CRS: EPSG:4326

  joined = gpd.sjoin(gdf,current_geometry,how="inner",predicate="intersects")
  0%|          | 1/7697 [00:00<1:17:59,  1.64it/s]C:\Users\Vitor\AppData\Local\Temp\ipykernel_11112\1367202915.py:11: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: OGC:CRS84
Right CRS: EPSG:4326

  joined = gpd.sjoin(gdf,current_geometry,how="inner",predicate="intersects")
  0%|          | 2/7697 [00:00<47:47,  2.68it/s]  C:\Users\Vitor\AppData\Local\Temp\ipykernel_11112\1367202915.py:11: UserWarning: CRS mismatch between the CRS of left ge

In [30]:
%%time

folder_path = r'E:\arquivos_modulo_5\car_mpb_desmat'

file_paths = os.listdir(folder_path)

CPU times: total: 31.2 ms
Wall time: 30 ms


In [ ]:
# Declarando como variável uma lista vazia
gdfs = []

In [33]:
import pandas as pd

# Inicia um loop
for file in tqdm(file_paths):
    # Formar o caminho completo do arquivo e abre como um geodataframe
    gdf_joined = gpd.read_file(os.path.join(folder_path,file))
    # Juntar cada geodataframe na lista gdfs
    gdfs.append(gdf_joined)
# Concatenação de todos os geodataframes
combined_gdf = gpd.GeoDataFrame(pd.concat(gdfs,ignore_index=True))
# Seleciono o sistema de coordenadas de um dos arquiuvos de entrada como o sistema de coordenadas do arquivo de saida
combined_gdf.crs = gdfs[0].crs
    

100%|██████████| 7102/7102 [07:41<00:00, 15.40it/s]


In [34]:
print(combined_gdf.head())

      fid  DN                                           geometry
0  544764  39  POLYGON ((-62.01529 -11.93861, -62.01529 -11.9...
1  556199  39  POLYGON ((-62.16890 -12.05907, -62.16890 -12.0...
2  554887  39  POLYGON ((-62.17024 -12.04371, -62.17024 -12.0...
3  554385  39  POLYGON ((-62.01313 -12.03644, -62.01313 -12.0...
4  572907  39  POLYGON ((-62.13521 -12.21457, -62.13521 -12.2...


In [35]:
combined_gdf = combined_gdf.to_crs(epsg=32721)

In [36]:
print(combined_gdf.crs)

EPSG:32721


In [41]:
# Dissolve das geometrias para desconsiderar as sobreposições
combined_gdf = combined_gdf.dissolve(by=None)
# Calculo da área da geoemtria dissolvida
combined_gdf['area_ha'] = combined_gdf.area / 10000

In [38]:
print(combined_gdf.head())

      fid  DN                                           geometry   area_ha
0  544764  39  POLYGON ((-46725.668 8675275.449, -46725.125 8...  1.761844
1  556199  39  POLYGON ((-63261.020 8661595.318, -63259.890 8...  0.440470
2  554887  39  POLYGON ((-63440.435 8663297.526, -63439.871 8...  0.440496
3  554385  39  POLYGON ((-46292.299 8664424.075, -46290.110 8...  1.673141
4  572907  39  POLYGON ((-59254.943 8644406.205, -59254.374 8...  1.408545


In [42]:
# Soma as áreas dissolvidas
print(combined_gdf['area_ha'].sum())

276763.03547399834


In [43]:
# Cálculo da produção total
(combined_gdf['area_ha'].sum())*70

19373412.483179886

In [44]:
combined_gdf.to_file(r'E:\arquivos_modulo_5\car_mpb_desmat\gdffinal.geojson',driver='GeoJSON')